In [0]:
# Databricks notebook source
# MAGIC %md
# MAGIC # PDF Ingestion and Chunking
# MAGIC Reads PDFs from a Unity Catalog volume, extracts text, splits it into chunks,
# MAGIC and writes the chunks to a Delta table.


In [0]:
%pip install pypdf --quiet

In [0]:


# COMMAND ----------

dbutils.library.restartPython()

In [0]:
dbutils.widgets.text("catalog", "dev_catalog")
dbutils.widgets.text("schema", "docs")
dbutils.widgets.text("volume", "raw_pdfs")
dbutils.widgets.text("target_table", "pdf_chunks")
dbutils.widgets.text("chunk_size", "1000")       # characters per chunk
dbutils.widgets.text("chunk_overlap", "200")     # overlap between chunks

CATALOG = dbutils.widgets.get("catalog")
SCHEMA = dbutils.widgets.get("schema")
VOLUME = dbutils.widgets.get("volume")
TARGET_TABLE = dbutils.widgets.get("target_table")
CHUNK_SIZE = int(dbutils.widgets.get("chunk_size"))
CHUNK_OVERLAP = int(dbutils.widgets.get("chunk_overlap"))

VOLUME_PATH = f"/Volumes/{CATALOG}/{SCHEMA}/{VOLUME}"
FULL_TABLE_NAME = f"{CATALOG}.{SCHEMA}.{TARGET_TABLE}"

print(f"Reading PDFs from: {VOLUME_PATH}")
print(f"Writing chunks to: {FULL_TABLE_NAME}")

In [0]:
/Volumes/rag_chatbot_dev/bronze/docs

In [0]:



import hashlib
import os
from datetime import datetime, timezone

from pypdf import PdfReader
from pyspark.sql import Row
from pyspark.sql import functions as F
from pyspark.sql.types import (
    IntegerType,
    StringType,
    StructField,
    StructType,
    TimestampType,
)


def list_pdf_files(root_path: str) -> list[str]:
    """Recursively list all PDF files under a volume path, as local paths."""
    pdf_files = []
    for dirpath, _, filenames in os.walk(root_path):
        for name in filenames:
            if name.lower().endswith(".pdf"):
                pdf_files.append(os.path.join(dirpath, name))
    return pdf_files


def extract_pages(pdf_path: str) -> list[tuple[int, str]]:
    """Return a list of (page_number, text) tuples. Page numbers start at 1."""
    # Volumes are mounted as a local filesystem, so PdfReader can open them directly
    reader = PdfReader(pdf_path)
    pages = []
    for i, page in enumerate(reader.pages, start=1):
        text = page.extract_text() or ""
        pages.append((i, text.strip()))
    return pages


def chunk_text(text: str, chunk_size: int, overlap: int) -> list[str]:
    """Split text into fixed-size character chunks with overlap.
    Tries to break on whitespace so words are not cut in half."""
    if chunk_size <= overlap:
        raise ValueError("chunk_size must be greater than chunk_overlap")

    text = " ".join(text.split())  # normalize whitespace
    if not text:
        return []

    chunks = []
    start = 0
    n = len(text)
    while start < n:
        end = min(start + chunk_size, n)
        # Back up to the last space so we don't split a word
        if end < n:
            space_idx = text.rfind(" ", start, end)
            if space_idx > start:
                end = space_idx
        chunk = text[start:end].strip()
        if chunk:
            chunks.append(chunk)
        if end >= n:
            break
        # Move forward, keeping the overlap
        start = max(end - overlap, start + 1)
    return chunks


def make_chunk_id(file_path: str, page: int, idx: int) -> str:
    """Deterministic ID so reruns produce the same key for the same chunk."""
    raw = f"{file_path}|{page}|{idx}"
    return hashlib.sha256(raw.encode("utf-8")).hexdigest()

# COMMAND ----------

pdf_paths = list_pdf_files(VOLUME_PATH)
print(f"Found {len(pdf_paths)} PDF files")

records = []
ingested_at = datetime.now(timezone.utc)

for pdf_path in pdf_paths:
    try:
        pages = extract_pages(pdf_path)
    except Exception as e:
        print(f"Skipping {pdf_path}: {e}")
        continue

    file_name = os.path.basename(pdf_path)
    chunk_idx = 0

    for page_number, page_text in pages:
        for chunk in chunk_text(page_text, CHUNK_SIZE, CHUNK_OVERLAP):
            records.append(
                Row(
                    chunk_id=make_chunk_id(pdf_path, page_number, chunk_idx),
                    file_name=file_name,
                    file_path=pdf_path,
                    page_number=page_number,
                    chunk_index=chunk_idx,
                    chunk_text=chunk,
                    char_count=len(chunk),
                    ingested_at=ingested_at,
                )
            )
            chunk_idx += 1

print(f"Created {len(records)} chunks")

# COMMAND ----------

schema = StructType([
    StructField("chunk_id", StringType(), False),
    StructField("file_name", StringType(), True),
    StructField("file_path", StringType(), True),
    StructField("page_number", IntegerType(), True),
    StructField("chunk_index", IntegerType(), True),
    StructField("chunk_text", StringType(), True),
    StructField("char_count", IntegerType(), True),
    StructField("ingested_at", TimestampType(), True),
])

if records:
    df = spark.createDataFrame(records, schema=schema)
else:
    df = spark.createDataFrame([], schema=schema)

# COMMAND ----------

# Write to a Delta table. Use MERGE so re-running the notebook doesn't duplicate chunks.
df.createOrReplaceTempView("new_chunks")

spark.sql(f"""
    CREATE TABLE IF NOT EXISTS {FULL_TABLE_NAME} (
        chunk_id     STRING NOT NULL,
        file_name    STRING,
        file_path    STRING,
        page_number  INT,
        chunk_index  INT,
        chunk_text   STRING,
        char_count   INT,
        ingested_at  TIMESTAMP
    )
    USING DELTA
    TBLPROPERTIES (delta.enableChangeDataFeed = true)
""")

spark.sql(f"""
    MERGE INTO {FULL_TABLE_NAME} AS target
    USING new_chunks AS source
    ON target.chunk_id = source.chunk_id
    WHEN MATCHED THEN UPDATE SET *
    WHEN NOT MATCHED THEN INSERT *
""")

print(f"Upsert complete into {FULL_TABLE_NAME}")

# COMMAND ----------

display(spark.table(FULL_TABLE_NAME).groupBy("file_name").count().orderBy("file_name"))

In [0]:
%sql
select * from rag_chatbot_dev.bronze.rag_chatbot